In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [2]:
# Datasets & DataLoaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),     # image scale (0,1)
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # normalize (-1,1)  (std)(mean)
])

trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

100%|██████████| 170M/170M [02:22<00:00, 1.19MB/s] 


In [3]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [4]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size = 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # kernel_size = 2, stride = 2

            nn.Conv2d(32, 64, kernel_size = 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # kernel_size = 2, stride = 2

            nn.Conv2d(64, 128, kernel_size = 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # kernel_size = 2, stride = 2
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),

            nn.Linear(256, 10),
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1) # flattening
        x = self.fc_layers(x)

        return x

In [6]:
model = CNN()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.parameters(),
    weight_decay=0.001
)

In [7]:
# Training the model

train_losses = []
val_losses = []

best_val_loss = float('inf')

epochs = 10

for epoch in range(epochs):
    # -------------------
    # TRAINING PHASE
    # -------------------
    model.train()
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        outputs = model(images)   # no need for .forward()
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    train_loss = epoch_training_loss / len(trainloader)
    train_losses.append(train_loss)

    # -------------------
    # VALIDATION PHASE
    # -------------------
    model.eval()
    epoch_val_loss = 0.0

    with torch.no_grad():
        for images, labels in testloader:   # make sure you have valloader
            outputs = model(images)
            loss = criterion(outputs, labels)

            epoch_val_loss += loss.item()

    val_loss = epoch_val_loss / len(testloader)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f}, Val Loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), 'best_model.pt')


Epoch 1: Train Loss=1.4092, Val Loss=1.0672
Epoch 2: Train Loss=0.9913, Val Loss=0.9711
Epoch 3: Train Loss=0.8184, Val Loss=0.7881
Epoch 4: Train Loss=0.7228, Val Loss=0.7782
Epoch 5: Train Loss=0.6512, Val Loss=0.7368
Epoch 6: Train Loss=0.5964, Val Loss=0.7391
Epoch 7: Train Loss=0.5448, Val Loss=0.7113
Epoch 8: Train Loss=0.5044, Val Loss=0.7123
Epoch 9: Train Loss=0.4630, Val Loss=0.7116
Epoch 10: Train Loss=0.4271, Val Loss=0.6906


In [ ]:
import matplotlib.pyplot as plt

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")

plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.title("Training vs Validation Loss")

plt.legend()
plt.show()

In [ ]:
# Evaluate our Model

correct_labels = 0
total_labels = 0

model.eval()
with torch.no_grad():
    for images, labels in testloader:
        output = model.forward(images)
        _,predicted  = torch.max(output, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")